<a href="https://colab.research.google.com/github/najeeb160/explainable-nids/blob/main/notebooks/Phase0_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 0 — Environment Setup & LLM PoC**
### Gate criteria (all must pass):

1. GPU visible (T4)
2. GraphIDS cloned, commit hash recorded
3. uv sync clean
4. Dataset extracted and readable
5. main.py --help works
6. Groq PoC generates an explanation, saved to Drive
7. Restart & Run All → zero errors

In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
assert torch.cuda.is_available(), "❌ No GPU. Runtime → Change runtime type → T4 GPU, then rerun."

Thu Oct 29 06:17:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/lorenzo9uerra/GraphIDS.git
%cd GraphIDS
COMMIT_HASH = !git rev-parse HEAD
print("Pinned GraphIDS commit:", COMMIT_HASH[0])

fatal: destination path 'GraphIDS' already exists and is not an empty directory.
/content/GraphIDS
Pinned GraphIDS commit: 2feac6a369e9acfd660b6faf7c38100d7dd13453


In [3]:
!pip install -q uv
!uv sync

Resolved 85 packages in 1ms
Checked 75 packages in 0.82ms


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os, glob

DRIVE_PROJECT = '/content/drive/MyDrive/NIDS_Project'
os.makedirs(DRIVE_PROJECT, exist_ok=True)

zips = glob.glob(f'{DRIVE_PROJECT}/*.zip')
print("Found zips:", zips)
assert len(zips) == 1, f"Expected exactly 1 zip in {DRIVE_PROJECT}, found {len(zips)}: {zips}"

os.makedirs('data', exist_ok=True)
!unzip -d data/NF-UNSW-NB15-v3 -j "{zips[0]}"

csv_path = 'data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv'
assert os.path.exists(csv_path), "❌ CSV missing after unzip — check output above."
!ls -lh data/NF-UNSW-NB15-v3/

Found zips: ['/content/drive/MyDrive/NIDS_Project/f7546561558c07c5_NFV3DATA-A11964_A11964.zip']
Archive:  /content/drive/MyDrive/NIDS_Project/f7546561558c07c5_NFV3DATA-A11964_A11964.zip
replace data/NF-UNSW-NB15-v3/FurtherInformation.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
total 551M
-rw-rw-rw- 1 root root  839 Feb 28  2025 bag-info.txt
-rw-rw-rw- 1 root root   55 Feb 28  2025 bagit.txt
-rw-rw-rw- 1 root root  143 Feb 28  2025 FurtherInformation.txt
-rw-rw-rw- 1 root root  136 Feb 28  2025 manifest-sha1.txt
-rw-rw-rw- 1 root root 2.9K Feb 28  2025 NetFlow_v3_Features.csv
-rw-rw-rw- 1 root root 551M Feb 28  2025 NF-UNSW-NB15-v3.csv
-rw-rw-rw- 1 root root  164 Feb 28  2025 tagmanifest-sha1.txt


In [6]:
!uv run main.py --help

usage: main.py [-h] [--config CONFIG] --data_dir DATA_DIR
               [--checkpoint CHECKPOINT] [--seed SEED]
               [--dataset {NF-UNSW-NB15-v3,NF-CSE-CIC-IDS2018-v3,NF-UNSW-NB15-v2,NF-CSE-CIC-IDS2018-v2}]
               [--fanout FANOUT] [--edim_out EDIM_OUT]
               [--ae_embedding_dim AE_EMBEDDING_DIM]
               [--ae_feedforward_dim AE_FEEDFORWARD_DIM] [--dropout DROPOUT]
               [--ae_dropout AE_DROPOUT] [--learning_rate LEARNING_RATE]
               [--weight_decay WEIGHT_DECAY]
               [--ae_weight_decay AE_WEIGHT_DECAY] [--num_epochs NUM_EPOCHS]
               [--fraction FRACTION]
               [--split_mode {stratified,temporal,temporal_shift_aware}]
               [--distribution_segment {pre_shift,post_shift}]
               [--patience PATIENCE]
               [--positional_encoding {None,learnable,sinusoidal}]
               [--agg_type {mean}] [--num_layers NUM_LAYERS]
               [--mask_ratio MASK_RATIO] [--step_percent STEP_PE

In [7]:
import pandas as pd
df = pd.read_csv('data/NF-UNSW-NB15-v3/NF-UNSW-NB15-v3.csv', nrows=1000)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (1000, 55)
Columns: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG'

,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1424242193040,1424242193043,59.166.0.2,4894,149.171.126.3,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
1,1424242192744,1424242193079,59.166.0.4,52671,149.171.126.6,31992,6,11.0,4704,28,...,0,91,12,19,0,90,12,19,0,Benign
2,1424242190649,1424242193109,59.166.0.0,47290,149.171.126.9,6881,6,37.0,13662,238,...,0,1843,10,119,0,1843,5,88,0,Benign
3,1424242193145,1424242193146,59.166.0.8,43310,149.171.126.7,53,17,5.0,146,2,...,0,0,0,0,0,0,0,0,0,Benign
4,1424242193239,1424242193241,59.166.0.1,45870,149.171.126.1,53,17,5.0,130,2,...,0,0,0,0,0,0,0,0,0,Benign


In [8]:
!pip install -q groq
from getpass import getpass
from groq import Groq

api_key = getpass("Paste your Groq API key: ")
client = Groq(api_key=api_key)
print("Groq client ready ✅")

Paste your Groq API key: ··········
Groq client ready ✅


In [9]:
models = client.models.list()
model_ids = sorted(m.id for m in models.data)
print(model_ids)
assert any("gpt-oss-20b" in m for m in model_ids), "❌ gpt-oss-20b not available on this key"

['allam-2-7b', 'canopylabs/orpheus-arabic-saudi', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'openai/gpt-oss-safeguard-20b', 'qwen/qwen3.8-27b', 'whisper-large-v3', 'whisper-large-v3-turbo']


In [10]:
from IPython.display import Markdown, display

flow = ("IPV4_SRC_ADDR: 192.168.1.5, IPV4_DST_ADDR: 10.0.0.7, "
        "L4_SRC_PORT: 443, L4_DST_PORT: 8080, PROTOCOL: 6, L7_PROTO: 7, "
        "IN_BYTES: 45236, OUT_BYTES: 812, IN_PKTS: 312, OUT_PKTS: 4, "
        "FLOW_DURATION_MILLISECONDS: 1200")

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "You are a network security analyst. Analyze this NetFlow and explain in plain language whether it looks suspicious and why. Be honest about uncertainty."},
        {"role": "user", "content": flow},
    ],
    temperature=0.1,
)
display(Markdown(response.choices[0].message.content))

**What the NetFlow record tells us**

| Field | Value | What it means |
|-------|-------|---------------|
| **Source IP** | 192.168.1.5 | A private‑network address (likely a workstation or server inside your LAN). |
| **Destination IP** | 10.0.0.7 | Another private‑network address (probably a server or appliance inside your LAN). |
| **Source port** | 443 | Normally the port used for outbound HTTPS traffic. |
| **Destination port** | 8080 | Commonly used for HTTP proxies, web servers, or application‑level gateways. |
| **Protocol** | 6 | TCP. |
| **Layer‑7 protocol** | 7 | The NetFlow exporter’s “L7_PROTO” value 7 isn’t a standard HTTP code; it usually means “unknown” or “other.” |
| **Bytes sent (source → dest)** | 45 236 | Roughly 45 KB of data flowed from 192.168.1.5 to 10.0.0.7. |
| **Bytes received (dest → source)** | 812 | About 0.8 KB came back. |
| **Packets sent** | 312 | 312 packets from the source to the destination. |
| **Packets received** | 4 | Only 4 packets came back. |
| **Duration** | 1 200 ms | The whole exchange lasted about 1.2 seconds. |

**Why this looks a little odd**

1. **Source port 443** – That port is normally used *by the client* to initiate HTTPS connections to external servers. Using it as the *source* port for a connection to an internal server on port 8080 is unusual. It could be a mis‑configured client, a deliberate attempt to disguise the traffic, or a sign of port‑spoofing.

2. **Large amount of data sent, very little data returned** – The client pushed ~45 KB of data to the server, but the server sent back only ~0.8 KB. That pattern is typical of a client uploading something (e.g., a file, a POST request, or a data‑exfiltration attempt) rather than downloading.

3. **Many packets in one direction, few in the other** – 312 packets from the client vs. only 4 from the server suggests the client is sending many small packets (perhaps a burst of requests or a scan) and the server is replying with a single small response.

4. **Layer‑7 protocol unknown** – If the exporter had correctly identified HTTP, we would expect a value of 1. A value of 7 means the exporter couldn’t classify the traffic, which can happen with encrypted or custom protocols.

**Is it suspicious?**

- **Not automatically malicious** – The traffic could be a legitimate upload to an internal web service or proxy. For example, a workstation might be sending a large file to an internal backup server that listens on 8080, and the server replies with a small acknowledgment.

- **Red flags that warrant a closer look** –  
  * The use of port 443 as the source port.  
  * The heavy upload relative to the download.  
  * The unknown L7 protocol.  

If you have a baseline of normal traffic for 192.168.1.5 and 10.0.0.7, compare this flow to that baseline. If this pattern is new or deviates from typical usage, investigate further.

**What to check next**

1. **Identify the hosts** – Confirm what 192.168.1.5 and 10.0.0.7 actually are (workstation, server, proxy, etc.).  
2. **Inspect the application** – If possible, look at the application logs on 10.0.0.7 to see what request it received and what response it sent.  
3. **Verify port usage** – Check whether 10.0.0.7 is supposed to accept traffic on port 8080 from internal clients.  
4. **Look for patterns** – Are there many similar flows from the same source? Are there repeated attempts with different source ports?  
5. **Check for encryption** – If the traffic is HTTPS (unlikely because port 8080 is not standard for HTTPS), the payload would be encrypted, making it harder to interpret.

**Bottom line**

The flow is **not obviously malicious**, but it does contain several atypical characteristics that merit a deeper look. If the traffic matches a known, legitimate use case in your environment, it’s probably fine. If it’s out of the ordinary, investigate the source host, the destination service, and the application logs to rule out abuse or compromise.

In [11]:
import datetime, json

poc_path = f'{DRIVE_PROJECT}/poc_llm_explanation.md'
with open(poc_path, 'w') as f:
    f.write("# Phase 0 — LLM PoC\n")
    f.write(f"- Model: openai/gpt-oss-20b\n")
    f.write(f"- Date: {datetime.datetime.now().isoformat()}\n")
    f.write(f"- GraphIDS commit: {COMMIT_HASH[0]}\n\n---\n\n")
    f.write(response.choices[0].message.content)
print(f"PoC saved to {poc_path} ✅")

manifest = {
    "date": datetime.datetime.now().isoformat(),
    "graphids_commit": COMMIT_HASH[0],
    "llm_model": "openai/gpt-oss-20b",
    "gpu": torch.cuda.get_device_name(0),
    "dataset": "NF-UNSW-NB15-v3",
}
with open(f'{DRIVE_PROJECT}/phase0_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print("Manifest saved ✅")

PoC saved to /content/drive/MyDrive/NIDS_Project/poc_llm_explanation.md ✅
Manifest saved ✅


✅ Gate 0 Checklist  
 [✔] Ran top-to-bottom with zero errors (Runtime → Restart and run all)  
 [✔] GPU asserted visible  
 [✔] GraphIDS commit hash printed  
 [✔] CSV loaded: 1000 rows × 55 columns  
 [✔] LLM explanation displayed  
 [✔] PoC + manifest saved to Drive